#  02: Data Preprocessing

This notebook prepares the cleaned dataset (from `01_EDA.ipynb`) for machine learning.

**Goals of this notebook:**
1. Load the cleaned dataset
2. Engineer a few useful new features
3. Encode categorical variables (binary + one-hot)
4. Define feature sets and targets for **regression** and **classification**
5. Split data into training and testing sets
6. Scale numerical features (fit on train only — no data leakage)
7. Save fully processed datasets for modeling in `03_Modeling.ipynb`

---

**Input:** `data/processed/student_clean.csv`
**Outputs:**
- `data/processed/train_processed.csv`
- `data/processed/test_processed.csv`


## 1. Setup & Imports

In [1]:
# Core libraries
import pandas as pd
import numpy as np

# sklearn preprocessing tools
from sklearn.model_selection import train_test_split

# Allow imports from the src/ folder
import sys
sys.path.append("..")

from src.preprocessing import encode_features, scale_features

pd.set_option("display.max_columns", 60)


## 2. Load the Cleaned Dataset

In [2]:
df = pd.read_csv("../data/processed/student_clean.csv")

print(f"Shape: {df.shape}")
df.head()

Shape: (395, 34)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,pass_fail
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6,5,6,6,0
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4,5,5,6,0
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10,7,8,10,1
3,GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,yes,yes,yes,yes,yes,yes,3,2,2,1,1,5,2,15,14,15,1
4,GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,yes,no,yes,yes,no,no,4,3,2,1,2,5,4,6,10,10,1


## 3. Feature Engineering

Based on the EDA, we'll create a few new features that may help our models capture patterns more effectively:

| New Feature | Description | Rationale |
|---|---|---|
| `avg_prev_grade` | Average of `G1` and `G2` | Smooths noise between the two earlier grades into a single strong predictor |
| `total_alcohol` | Sum of `Dalc` + `Walc` | Combines weekday/weekend alcohol use into one lifestyle indicator |
| `parent_edu_avg` | Average of `Medu` and `Fedu` | Single measure of overall parental education level |


In [3]:
# Create new engineered features
df["avg_prev_grade"] = (df["G1"] + df["G2"]) / 2
df["total_alcohol"] = df["Dalc"] + df["Walc"]
df["parent_edu_avg"] = (df["Medu"] + df["Fedu"]) / 2

print(" New features created: 'avg_prev_grade', 'total_alcohol', 'parent_edu_avg'")
df[["G1", "G2", "avg_prev_grade", "Dalc", "Walc", "total_alcohol", "Medu", "Fedu", "parent_edu_avg"]].head()

 New features created: 'avg_prev_grade', 'total_alcohol', 'parent_edu_avg'


,G1,G2,avg_prev_grade,Dalc,Walc,total_alcohol,Medu,Fedu,parent_edu_avg
0,5,6,5.5,1,1,2,4,4,4.0
1,5,5,5.0,1,1,2,1,1,1.0
2,7,8,7.5,2,3,5,1,1,1.0
3,15,14,14.5,1,1,2,4,2,3.0
4,6,10,8.0,1,2,3,3,3,3.0


## 4. Encode Categorical Features

convert categorical columns:

- **Binary columns** (e.g., `schoolsup`, `internet` — values `yes`/`no`) → mapped to `1`/`0`
- **Multi-category columns** (e.g., `Mjob`, `Fjob`, `reason`) → **one-hot encoded** using `pd.get_dummies()` with `drop_first=True` to avoid multicollinearity (the "dummy variable trap")

This logic is encapsulated in `src/preprocessing.py` → `encode_features()` for reusability and clean code.

In [4]:
# Encode all categorical features
df_encoded = encode_features(df)

print(f"\nShape before encoding: {df.shape}")
print(f"Shape after encoding : {df_encoded.shape}")
df_encoded.head()

 Encoding complete. Dataset now has 46 columns.

Shape before encoding: (395, 37)
Shape after encoding : (395, 46)


,age,Medu,Fedu,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,pass_fail,avg_prev_grade,total_alcohol,parent_edu_avg,school_MS,sex_M,address_U,famsize_LE3,Pstatus_T,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_home,reason_other,reason_reputation,guardian_mother,guardian_other
0,18,4,4,2,2,0,1,0,0,0,1,1,0,0,4,3,4,1,1,3,6,5,6,6,0,5.5,2,4.0,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False
1,17,1,1,1,2,0,0,1,0,0,0,1,1,0,5,3,3,1,1,3,4,5,5,6,0,5.0,2,1.0,False,False,True,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False
2,15,1,1,1,2,3,1,0,1,0,1,1,1,0,4,3,2,2,3,3,10,7,8,10,1,7.5,5,1.0,False,False,True,True,True,False,False,False,False,False,True,False,False,False,True,False,True,False
3,15,4,2,1,3,0,0,1,1,1,1,1,1,1,3,2,2,1,1,5,2,15,14,15,1,14.5,2,3.0,False,False,True,False,True,True,False,False,False,False,False,True,False,True,False,False,True,False
4,16,3,3,1,2,0,0,1,1,0,1,1,0,0,4,3,2,1,2,5,4,6,10,10,1,8.0,3,3.0,False,False,True,False,True,False,True,False,False,False,True,False,False,True,False,False,False,False


In [5]:
# Confirm there are no remaining non-numeric columns
non_numeric = df_encoded.select_dtypes(include=["object"]).columns.tolist()
print(f"Remaining non-numeric columns: {non_numeric if non_numeric else 'None '}")

Remaining non-numeric columns: None 


## 5. Define Feature Sets & Targets

This project supports **two prediction tasks**:

1. **Regression** — predict the exact final grade `G3` (0–20)
2. **Classification** — predict `pass_fail` (1=Pass, 0=Fail)

**Important:** To avoid trivial "leakage" where the model just learns `G3 = avg(G1, G2)`, we keep `G1` and `G2` as features (they are realistic predictors — available *before* the final exam) but we must **drop `G3`** from the feature set for both tasks, and also drop `pass_fail` when predicting `G3`, and vice versa.

In [6]:
# Define the two targets
y_reg = df_encoded["G3"]            # Regression target: final grade (0-20)
y_clf = df_encoded["pass_fail"]     # Classification target: Pass(1) / Fail(0)

# Define the shared feature set (drop both targets from features)
X = df_encoded.drop(columns=["G3", "pass_fail"])

print(f"Feature matrix shape : {X.shape}")
print(f"Regression target    : y_reg -> {y_reg.shape}")
print(f"Classification target: y_clf -> {y_clf.shape}")

print("\nFeature columns:")
print(X.columns.tolist())

Feature matrix shape : (395, 44)
Regression target    : y_reg -> (395,)
Classification target: y_clf -> (395,)

Feature columns:
['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'avg_prev_grade', 'total_alcohol', 'parent_edu_avg', 'school_MS', 'sex_M', 'address_U', 'famsize_LE3', 'Pstatus_T', 'Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher', 'Fjob_health', 'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_home', 'reason_other', 'reason_reputation', 'guardian_mother', 'guardian_other']


## 6. Train-Test Split

In [7]:
# Split features and both targets together, stratifying on the classification label
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set : {X_test.shape[0]} samples")

print("\nClass balance check (pass_fail):")
print("Train:", y_clf_train.value_counts(normalize=True).round(3).to_dict())
print("Test :", y_clf_test.value_counts(normalize=True).round(3).to_dict())

Training set: 316 samples
Testing set : 79 samples

Class balance check (pass_fail):
Train: {1: 0.671, 0: 0.329}
Test : {1: 0.671, 0: 0.329}


## 7. Feature Scaling

**Key principle — avoid data leakage:**
> The scaler is **fit only on the training data**, then used to **transform** both the training and test sets. This ensures the test set remains completely unseen during preprocessing.

We use `StandardScaler` (zero mean, unit variance), implemented in `src/preprocessing.py` → `scale_features()`.

In [8]:
# Scale features (fit on train, transform train & test)
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

# Convert back to DataFrames to preserve column names for readability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

X_train_scaled.head()

 Features scaled using StandardScaler.


,age,Medu,Fedu,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,avg_prev_grade,total_alcohol,parent_edu_avg,school_MS,sex_M,address_U,famsize_LE3,Pstatus_T,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_home,reason_other,reason_reputation,guardian_mother,guardian_other
140,-1.305332,1.135587,0.462701,0.722867,2.386735,-0.442589,2.554175,0.777214,-0.974996,-1.019171,0.518726,0.252929,0.448911,-0.715515,-2.091898,-1.272747,-1.038594,-0.554434,-1.02116,-0.408507,-0.688705,-1.205050,-0.465924,-0.841690,-0.906886,0.892650,-0.369717,1.019171,0.557856,-0.616371,0.329806,-0.317876,-0.740959,-0.587095,2.454022,-0.207134,-1.143053,1.648529,-0.286691,-0.591969,-0.323875,-0.621261,-1.572427,-0.280175
245,-0.524111,-0.667656,-1.388103,2.115708,-1.262660,-0.442589,-0.391516,-1.286648,-0.974996,-1.019171,0.518726,0.252929,0.448911,-0.715515,0.093357,-0.269784,-0.136957,-0.554434,-1.02116,0.312656,0.057458,2.159411,1.921620,2.113577,-0.906886,-1.136684,-0.369717,1.019171,0.557856,-0.616371,0.329806,-0.317876,1.349603,-0.587095,-0.407494,-0.207134,0.874850,-0.606601,-0.286691,-0.591969,-0.323875,-0.621261,0.635959,-0.280175
131,-1.305332,-1.569278,-1.388103,2.115708,-1.262660,-0.442589,-0.391516,0.777214,-0.974996,0.981190,-1.927801,0.252929,0.448911,1.397595,0.093357,-0.269784,-0.136957,-0.554434,-0.25650,0.312656,-0.688705,-0.899190,-2.853468,-2.023797,-0.417849,-1.644018,-0.369717,-0.981190,0.557856,-0.616371,0.329806,-0.317876,-0.740959,-0.587095,-0.407494,-0.207134,0.874850,-0.606601,-0.286691,-0.591969,-0.323875,-0.621261,0.635959,-0.280175
153,1.819554,0.233965,-0.462701,-0.669974,-1.262660,3.553358,-0.391516,0.777214,-0.974996,-1.019171,0.518726,-3.953679,0.448911,1.397595,0.093357,1.736141,0.764679,-0.554434,-1.02116,0.312656,-0.688705,-1.816770,-2.853468,-2.467087,-0.906886,-0.122017,-0.369717,1.019171,0.557856,-0.616371,0.329806,-0.317876,-0.740959,1.703301,-0.407494,-0.207134,-1.143053,-0.606601,-0.286691,1.689278,-0.323875,-0.621261,0.635959,-0.280175
386,1.038332,1.135587,1.388103,2.115708,-1.262660,-0.442589,-0.391516,0.777214,1.025645,0.981190,0.518726,0.252929,0.448911,1.397595,0.093357,0.733178,-0.136957,0.513866,-0.25650,1.033818,0.181818,-1.510910,-1.527054,-1.580507,0.071189,1.399984,2.704772,-0.981190,-1.792577,-0.616371,0.329806,-0.317876,-0.740959,-0.587095,2.454022,-0.207134,-1.143053,-0.606601,-0.286691,-0.591969,-0.323875,1.609630,0.635959,-0.280175


In [9]:
# Quick sanity check: scaled training data should have mean ~0 and std ~1
print("Training set (scaled) summary:")
print(X_train_scaled.describe().loc[["mean", "std"]].T.head(8))

Training set (scaled) summary:
                    mean       std
age         1.447506e-15  1.001586
Medu        2.529622e-17  1.001586
Fedu        8.432074e-18  1.001586
traveltime  1.054009e-16  1.001586
studytime  -2.501515e-16  1.001586
failures    3.934968e-17  1.001586
schoolsup   3.372829e-17  1.001586
famsup      3.583631e-17  1.001586


## 8. Assemble & Save Processed Datasets

In [10]:
# Combine scaled features with both targets for train and test sets
train_processed = X_train_scaled.copy()
train_processed["G3"] = y_reg_train.values
train_processed["pass_fail"] = y_clf_train.values

test_processed = X_test_scaled.copy()
test_processed["G3"] = y_reg_test.values
test_processed["pass_fail"] = y_clf_test.values

print(f"train_processed shape: {train_processed.shape}")
print(f"test_processed shape : {test_processed.shape}")

train_processed.head()

train_processed shape: (316, 46)
test_processed shape : (79, 46)


,age,Medu,Fedu,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,avg_prev_grade,total_alcohol,parent_edu_avg,school_MS,sex_M,address_U,famsize_LE3,Pstatus_T,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_home,reason_other,reason_reputation,guardian_mother,guardian_other,G3,pass_fail
140,-1.305332,1.135587,0.462701,0.722867,2.386735,-0.442589,2.554175,0.777214,-0.974996,-1.019171,0.518726,0.252929,0.448911,-0.715515,-2.091898,-1.272747,-1.038594,-0.554434,-1.02116,-0.408507,-0.688705,-1.205050,-0.465924,-0.841690,-0.906886,0.892650,-0.369717,1.019171,0.557856,-0.616371,0.329806,-0.317876,-0.740959,-0.587095,2.454022,-0.207134,-1.143053,1.648529,-0.286691,-0.591969,-0.323875,-0.621261,-1.572427,-0.280175,0,0
245,-0.524111,-0.667656,-1.388103,2.115708,-1.262660,-0.442589,-0.391516,-1.286648,-0.974996,-1.019171,0.518726,0.252929,0.448911,-0.715515,0.093357,-0.269784,-0.136957,-0.554434,-1.02116,0.312656,0.057458,2.159411,1.921620,2.113577,-0.906886,-1.136684,-0.369717,1.019171,0.557856,-0.616371,0.329806,-0.317876,1.349603,-0.587095,-0.407494,-0.207134,0.874850,-0.606601,-0.286691,-0.591969,-0.323875,-0.621261,0.635959,-0.280175,18,1
131,-1.305332,-1.569278,-1.388103,2.115708,-1.262660,-0.442589,-0.391516,0.777214,-0.974996,0.981190,-1.927801,0.252929,0.448911,1.397595,0.093357,-0.269784,-0.136957,-0.554434,-0.25650,0.312656,-0.688705,-0.899190,-2.853468,-2.023797,-0.417849,-1.644018,-0.369717,-0.981190,0.557856,-0.616371,0.329806,-0.317876,-0.740959,-0.587095,-0.407494,-0.207134,0.874850,-0.606601,-0.286691,-0.591969,-0.323875,-0.621261,0.635959,-0.280175,0,0
153,1.819554,0.233965,-0.462701,-0.669974,-1.262660,3.553358,-0.391516,0.777214,-0.974996,-1.019171,0.518726,-3.953679,0.448911,1.397595,0.093357,1.736141,0.764679,-0.554434,-1.02116,0.312656,-0.688705,-1.816770,-2.853468,-2.467087,-0.906886,-0.122017,-0.369717,1.019171,0.557856,-0.616371,0.329806,-0.317876,-0.740959,1.703301,-0.407494,-0.207134,-1.143053,-0.606601,-0.286691,1.689278,-0.323875,-0.621261,0.635959,-0.280175,0,0
386,1.038332,1.135587,1.388103,2.115708,-1.262660,-0.442589,-0.391516,0.777214,1.025645,0.981190,0.518726,0.252929,0.448911,1.397595,0.093357,0.733178,-0.136957,0.513866,-0.25650,1.033818,0.181818,-1.510910,-1.527054,-1.580507,0.071189,1.399984,2.704772,-0.981190,-1.792577,-0.616371,0.329806,-0.317876,-0.740959,-0.587095,2.454022,-0.207134,-1.143053,-0.606601,-0.286691,-0.591969,-0.323875,1.609630,0.635959,-0.280175,6,0


In [11]:
# Save processed datasets to disk
train_path = "../data/processed/train_processed.csv"
test_path = "../data/processed/test_processed.csv"

train_processed.to_csv(train_path, index=False)
test_processed.to_csv(test_path, index=False)

print(f" Saved: {train_path}")
print(f" Saved: {test_path}")


 Saved: ../data/processed/train_processed.csv
 Saved: ../data/processed/test_processed.csv


## 9. Summary

| Step | Outcome |
|---|---|
| **Feature Engineering** | Added `avg_prev_grade`, `total_alcohol`, `parent_edu_avg` |
| **Encoding** | Binary columns → 0/1, multi-category columns → one-hot encoded |
| **Targets Defined** | `G3` (regression) and `pass_fail` (classification) |
| **Train-Test Split** | 80/20 split, stratified on `pass_fail` for class balance |
| **Scaling** | `StandardScaler` fit on train only, applied to train & test |
| **Saved Files** | `train_processed.csv`, `test_processed.csv` |

---

### ➡️ Next Step
Proceed to **`03_Modeling.ipynb`** to train and evaluate multiple machine learning models (Logistic Regression, Random Forest, Gradient Boosting) for both regression and classification tasks.